# Timestral-209M — time-series forecasting on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/Timestral-209M-forecast.ipynb)

[`mistralai/Timestral-209M`](https://huggingface.co/mistralai/Timestral-209M) is Mistral's 209M **time-series foundation model** for zero-shot probabilistic forecasting. It is not a language model: there is no tokenizer, no chat template, and no `generate()`. You pass a numeric series and get a horizon of quantile forecasts back. The same family ships smaller checkpoints ([5M](https://huggingface.co/mistralai/Timestral-5M), [28M](https://huggingface.co/mistralai/Timestral-28M), [123M](https://huggingface.co/mistralai/Timestral-123M)); this notebook loads 209M and optionally compares it with 5M on the same CSV.

**Licence (read this):** the Hub tag is `other`. The **weights** are [Modified MIT](https://huggingface.co/mistralai/Timestral-209M/blob/main/LICENSE), not standard MIT and not Apache. Keep the attribution notice (`2026 - Mistral AI`). If your company's global consolidated monthly revenue exceeded **USD 20 million** for the preceding month, you are not authorized to use the weights (or derivatives) under this license — contact Mistral for a commercial license or use their hosted services. Do not use the model in a way that infringes, misappropriates, or otherwise violates a third party's rights. The `timestral` **inference code** on [GitHub](https://github.com/mistralai/timestral/) is Apache 2.0 and is a separate grant.

**Runtime:** *Runtime > Change runtime type > T4 GPU*. The free T4 has ~15 GB of VRAM. Timestral-209M is 837 MB of float32 weights (~209M parameters), so it also runs on CPU; this notebook prefers CUDA when a GPU is attached. There is **no bfloat16 requirement** and **no FlashAttention-2** path — install `timestral` and call `TimestralForecaster.from_pretrained(..., device=...)`.

**Card limits:** context up to **32,768** time steps (1,024 patches of 32). Horizon up to **1,024** steps in one forward pass (32 patches). Longer horizons are decoded block by block, feeding each block's median back as context. Outputs are **21 quantiles** (0.01, 0.05, 0.10, …, 0.95, 0.99).

**Links:** [209M card](https://huggingface.co/mistralai/Timestral-209M) · [5M card](https://huggingface.co/mistralai/Timestral-5M) · [collection](https://huggingface.co/collections/mistralai/timestral) · [GitHub](https://github.com/mistralai/timestral/) · [blog](https://mistral.ai/news/timestral/) · [weight license](https://huggingface.co/mistralai/Timestral-209M/blob/main/LICENSE)


In [ ]:
%pip install -q "git+https://github.com/mistralai/timestral.git@a534bdbd244d64f3d9d743f46c7bf6fc330ed4ff" pandas matplotlib

# The model card and the timestral README install the package from GitHub
# (Python >= 3.10, torch >= 2.4). The pin is the 2026-10-15 public release
# commit. pandas writes/reads the demo CSV; matplotlib draws history + bands.
# Colab already has torch, numpy, safetensors, and huggingface_hub.



> **Runtime tip:** pick a T4 if you have the choice. If `timestral` was imported before the install cell, use *Runtime > Restart session* and continue from the next cell. A fresh session does not need a restart.


In [ ]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("torch", torch.__version__)
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "(CPU is fine for this checkpoint)")
if DEVICE == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"VRAM total: {props.total_memory / 1024**3:.1f} GB  |  capability: sm_{props.major}{props.minor}")
    print("FlashAttention-2 is not used. Timestral runs on the default PyTorch kernels.")



## 1. Load Timestral-209M

`TimestralForecaster.from_pretrained` downloads `config.json` and `model.safetensors` (~837 MB) into the Hub cache on the first run. Inference was validated in float32; `from_pretrained` does not take a dtype argument.


In [ ]:
from timestral import TimestralForecaster

MODEL_209M = "mistralai/Timestral-209M"

forecaster = TimestralForecaster.from_pretrained(MODEL_209M, device=DEVICE)

n_params = sum(p.numel() for p in forecaster.model.parameters())
n_bytes = sum(p.numel() * p.element_size() for p in forecaster.model.parameters())
print("repo:", MODEL_209M)
print(f"parameters: {n_params:,} ({n_params / 1e6:.1f}M)")
print(f"weight memory: {n_bytes / 1024**2:.0f} MB  |  dtype: {next(forecaster.model.parameters()).dtype}")
print("device:", forecaster.device)
print("context_length:", forecaster.config.context_length, "| patch_size:", forecaster.patch_size)
print("max horizon in one pass:", forecaster.config.block_size * forecaster.patch_size, "steps")
print("quantile levels:", forecaster.quantile_levels)
if DEVICE == "cuda":
    print(f"VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"VRAM reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")



## 2. A CSV to forecast

The cells below write a synthetic daily series (trend + weekly seasonality + a few promo spikes) to `timestral_demo.csv`, then hold out the last 56 days. Swap the path for your own file: one date column and one numeric value column, regularly spaced. Mark missing observations as `NaN`.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

CSV_PATH = Path("timestral_demo.csv")
HORIZON = 56  # 8 weeks; well under the 1,024-step single-pass cap

rng = np.random.default_rng(7)
n = 730  # two years of daily observations
dates = pd.date_range("2024-01-01", periods=n, freq="D")
t = np.arange(n, dtype=np.float64)
weekly = 18.0 * np.sin(2 * np.pi * t / 7.0)
yearly = 12.0 * np.sin(2 * np.pi * t / 365.25)
trend = 0.045 * t
noise = rng.normal(0.0, 3.5, n)
values = 80.0 + trend + weekly + yearly + noise
for day in (59, 180, 364, 545):
    values[day] += 32.0
values = np.clip(values, 1.0, None)  # non-negative so clip_nonnegative (on by default) applies

pd.DataFrame({"date": dates, "value": values}).to_csv(CSV_PATH, index=False)
print("wrote", CSV_PATH.resolve(), f"({n} rows)")

frame = pd.read_csv(CSV_PATH, parse_dates=["date"])
history = frame["value"].to_numpy(dtype=np.float64)[:-HORIZON]
actual = frame["value"].to_numpy(dtype=np.float64)[-HORIZON:]
hist_dates = frame["date"].iloc[:-HORIZON]
fc_dates = frame["date"].iloc[-HORIZON:]
print(f"context: {len(history)} steps  |  holdout / prediction_length: {HORIZON}")
print(frame.tail(3).to_string(index=False))



## 3. Forecast with quantile bands

`forecast` accepts a 1-D series `(T,)`, a batch `(B, T)`, or multivariate `(B, V, T)`. The return value is a `Forecast`: `median` is the 0.5 quantile; `quantile(0.1)` / `quantile(0.9)` are the 10–90% band. Shapes on a single univariate series are `(1, 1, H)`.


In [ ]:
import matplotlib.pyplot as plt

forecast = forecaster.forecast(history, prediction_length=HORIZON)
median = forecast.median[0, 0].detach().cpu().numpy()
q05 = forecast.quantile(0.05)[0, 0].detach().cpu().numpy()
q10 = forecast.quantile(0.1)[0, 0].detach().cpu().numpy()
q90 = forecast.quantile(0.9)[0, 0].detach().cpu().numpy()
q95 = forecast.quantile(0.95)[0, 0].detach().cpu().numpy()

print("quantiles", tuple(forecast.quantiles.shape), "(series, variates, horizon, levels)")


def mae(y, yhat):
    return float(np.mean(np.abs(y - yhat)))


def mape(y, yhat):
    return float(np.mean(np.abs((y - yhat) / np.clip(np.abs(y), 1e-8, None))) * 100.0)


print(f"Timestral-209M  MAE {mae(actual, median):.3f}   MAPE {mape(actual, median):.2f}%")

fig, ax = plt.subplots(figsize=(12, 4.8))
ax.plot(hist_dates, history, color="#1f4e79", lw=1.4, label="history")
ax.plot(fc_dates, actual, color="#2ca02c", lw=1.6, label="holdout")
ax.plot(fc_dates, median, color="#d62728", lw=1.8, label="209M median")
ax.fill_between(fc_dates, q05, q95, color="#d62728", alpha=0.10, label="5–95% band")
ax.fill_between(fc_dates, q10, q90, color="#d62728", alpha=0.22, label="10–90% band")
ax.axvline(fc_dates.iloc[0], color="#666666", ls="--", lw=1)
ax.set_title("Timestral-209M — history, holdout, and quantile forecast")
ax.set_xlabel("date")
ax.set_ylabel("value")
ax.legend(loc="upper left", ncol=2, frameon=False)
ax.grid(True, alpha=0.25)
fig.tight_layout()
plt.show()



## 4. Optional: Timestral-5M on the same series

The 5M checkpoint is 21 MB. Load it next to 209M (both fit a T4 with room to spare) and compare median MAE / MAPE plus the visual fit. 209M usually wins on the public leaderboards; a single synthetic holdout is only a sanity check.


In [ ]:
MODEL_5M = "mistralai/Timestral-5M"

forecaster_5m = TimestralForecaster.from_pretrained(MODEL_5M, device=DEVICE)
n_5m = sum(p.numel() for p in forecaster_5m.model.parameters())
print(f"Timestral-5M parameters: {n_5m:,} ({n_5m / 1e6:.1f}M)  |  device: {forecaster_5m.device}")
if DEVICE == "cuda":
    print(f"VRAM allocated with both models: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

forecast_5m = forecaster_5m.forecast(history, prediction_length=HORIZON)
median_5m = forecast_5m.median[0, 0].detach().cpu().numpy()
q10_5m = forecast_5m.quantile(0.1)[0, 0].detach().cpu().numpy()
q90_5m = forecast_5m.quantile(0.9)[0, 0].detach().cpu().numpy()

rows = [
    ("Timestral-209M", median, q10, q90),
    ("Timestral-5M", median_5m, q10_5m, q90_5m),
]
print(f"{'model':<16} {'MAE':>8} {'MAPE':>8}")
for name, med, *_ in rows:
    print(f"{name:<16} {mae(actual, med):8.3f} {mape(actual, med):7.2f}%")

fig, ax = plt.subplots(figsize=(12, 4.8))
ax.plot(hist_dates, history, color="#1f4e79", lw=1.2, label="history")
ax.plot(fc_dates, actual, color="#2ca02c", lw=1.8, label="holdout")
ax.plot(fc_dates, median, color="#d62728", lw=1.8, label="209M median")
ax.fill_between(fc_dates, q10, q90, color="#d62728", alpha=0.16, label="209M 10–90%")
ax.plot(fc_dates, median_5m, color="#ff7f0e", lw=1.6, ls="--", label="5M median")
ax.fill_between(fc_dates, q10_5m, q90_5m, color="#ff7f0e", alpha=0.12, label="5M 10–90%")
ax.axvline(fc_dates.iloc[0], color="#666666", ls="--", lw=1)
ax.set_title("Timestral-209M vs Timestral-5M on the same holdout")
ax.set_xlabel("date")
ax.set_ylabel("value")
ax.legend(loc="upper left", ncol=2, frameon=False)
ax.grid(True, alpha=0.25)
fig.tight_layout()
plt.show()



## 5. Caveats

**Licence.** Weights: [Modified MIT](https://huggingface.co/mistralai/Timestral-209M/blob/main/LICENSE) (`license: other` on the Hub). Keep the attribution notice. The USD 20 million monthly-revenue restriction applies to the Model and any derivatives. The `timestral` package is Apache 2.0; that does not relicense the weights. *You must not use this model in a manner that infringes, misappropriates, or otherwise violates any third party's rights, including intellectual property rights.*

**Not an LLM.** Timestral is a patch-based time-series transformer. Do not use it for chat, code, classification of text, RAG, or any other language-model task. There is no prompt, no tokens, and no instruction following.

**Context and horizon (from the card).** History is capped at **32,768** steps; older points are dropped. A single forward pass forecasts at most **1,024** steps. Longer requests are filled block by block using the previous block's **median**, so uncertainty compounds. This notebook stays at 56 steps.

**Data shape.** Feed regularly spaced numeric values. Multivariate series are forecast jointly (`(B, V, T)`). Past-only and known-future covariates exist on the API (`past_covariates`, `known_covariates`, categorical variants) but are unused here. Missing values are `NaN`.

**Quantile bands.** The 21 knots are model quantiles, not a calibrated prediction interval for your series. A 10–90% band is not a promise that 80% of future points will land inside it.

**When not to rely on the demo numbers.** The CSV is synthetic. MAE / MAPE on one holdout window is a smoke test, not a benchmark. Validate on your own history, compare against a naive or seasonal-naive baseline, and do not use a single zero-shot pass for high-stakes inventory, pricing, or capacity decisions without that check.
